# 00 — Exploratory Data Analysis

**Runtime:** CPU only. Tidak butuh GPU.

**Tujuan:** Pahami dataset sebelum training. Notebook ini harus dijalankan pertama kali dan tidak mengubah data apapun.

**Yang dianalisis:**
1. Distribusi visibility keypoint (v=0/1/2)
2. Frekuensi kemunculan per keypoint (class imbalance)
3. Visualisasi canonical pitch dengan 32 keypoints
4. Contoh frame dengan overlay keypoint
5. Distribusi gambar per source (broadcast vs scouting)
6. Dataset split statistics

In [ ]:
# Cell 1: Install dependencies
!pip install mplsoccer opencv-python-headless matplotlib numpy pandas tqdm --quiet
print('Dependencies ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Environment & Path Setup — kompatibel Kaggle DAN Colab (auto-detect)
# ══════════════════════════════════════════════════════════════════════
import os, sys

def _detect_env():
    if os.path.exists('/kaggle/working'):
        return 'kaggle'
    if os.path.exists('/content'):
        return 'colab'
    return 'local'

ENV = _detect_env()
print(f'Environment terdeteksi: {ENV}')

# ⚠️ GANTI dengan URL repo GitHub kamu sendiri (isi SEKALI saja per notebook)
GITHUB_REPO = 'https://github.com/atangorp/soccer-homography-benchmark.git'

def _link(src, dst):
    """Symlink src -> dst kalau src ada dan dst belum ada. Aman dipanggil berkali-kali."""
    if os.path.exists(src) and not os.path.exists(dst):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        os.symlink(src, dst)

if ENV == 'kaggle':
    PROJECT_ROOT = '/kaggle/working/soccer-homography-benchmark'
    if not os.path.exists(PROJECT_ROOT):
        os.system(f'git clone -q {GITHUB_REPO} "{PROJECT_ROOT}"')
    sys.path.insert(0, PROJECT_ROOT)

    # WAJIB: attach dataset "soccer-field-raw" via "+ Add Input" di sidebar kanan
    # (ganti 'soccer-field-raw' kalau slug dataset kamu berbeda)
    _link('/kaggle/input/soccer-field-raw',
          f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8')

elif ENV == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/soccer-homography-benchmark'
    sys.path.insert(0, PROJECT_ROOT)

else:
    PROJECT_ROOT = os.getcwd()
    sys.path.insert(0, PROJECT_ROOT)

YOLO_DIR = f'{PROJECT_ROOT}/data/raw/soccer-field-localization.v9i.yolov8'
print(f'PROJECT_ROOT : {PROJECT_ROOT}')


In [ ]:
# Cell 2: Import src utilities
from src.geometry.pitch_reference import (
    get_dst_dataframe, get_keypoint_names,
    PITCH_WIDTH, PITCH_HEIGHT, N_KEYPOINTS
)
from src.geometry.metrics import classify_image_source

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import glob

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
df_dst = get_dst_dataframe()
kpt_names = get_keypoint_names()
print(f'Pitch reference: {N_KEYPOINTS} keypoints on {PITCH_WIDTH}x{PITCH_HEIGHT} canvas')

In [ ]:
# Cell 3: Dataset split statistics
print('Dataset split statistics:\n')
print(f'{"Split":<8} {"Images":>8} {"Labels":>8} {"Broadcast":>12} {"Scouting":>10} {"Unknown":>9}')
print('-' * 58)

total_stats = {}
for split in ['train', 'val', 'test']:
    img_dir = f'{YOLO_DIR}/{split}/images'
    lbl_dir = f'{YOLO_DIR}/{split}/labels'
    imgs = glob.glob(f'{img_dir}/*.jpg') + glob.glob(f'{img_dir}/*.png')
    lbls = glob.glob(f'{lbl_dir}/*.txt')
    sources = {'broadcast': 0, 'scouting': 0, 'unknown': 0}
    for p in imgs:
        src = classify_image_source(os.path.basename(p))
        sources[src] += 1
    total_stats[split] = {'n_images': len(imgs), 'n_labels': len(lbls), **sources}
    print(f'{split:<8} {len(imgs):>8} {len(lbls):>8} {sources["broadcast"]:>12} {sources["scouting"]:>10} {sources["unknown"]:>9}')

total_imgs = sum(v['n_images'] for v in total_stats.values())
print(f'{"TOTAL":<8} {total_imgs:>8}')
print(f'\nN keypoints per instance: {N_KEYPOINTS}')

In [ ]:
# Cell 4: Visualisasi canonical pitch dengan 32 keypoints
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

# Gambar lapangan sederhana
ax.set_facecolor('#3a7d44')  # hijau lapangan
ax.set_xlim(-5, PITCH_WIDTH + 5)
ax.set_ylim(-5, PITCH_HEIGHT + 5)

# Garis lapangan utama
for line in [
    # Batas lapangan
    [(0,0),(120,0)], [(120,0),(120,80)], [(120,80),(0,80)], [(0,80),(0,0)],
    # Garis tengah
    [(60,0),(60,80)],
    # Kotak penalti kiri
    [(0,18),(18,18)], [(18,18),(18,62)], [(18,62),(0,62)],
    # Kotak penalti kanan
    [(120,18),(102,18)], [(102,18),(102,62)], [(102,62),(120,62)],
    # Garis gawang kiri
    [(0,30),(6,30)], [(6,30),(6,50)], [(6,50),(0,50)],
    # Garis gawang kanan
    [(120,30),(114,30)], [(114,30),(114,50)], [(114,50),(120,50)],
]:
    ax.plot([line[0][0], line[1][0]], [line[0][1], line[1][1]],
            'white', linewidth=1.5, alpha=0.8)

# Lingkaran tengah
circle = plt.Circle((60, 40), 10, fill=False, color='white', linewidth=1.5, alpha=0.8)
ax.add_patch(circle)

# Plot 32 keypoints
for _, row in df_dst.iterrows():
    kpt_id = int(row['kpt_id'])
    x, y   = row['x'], row['y']
    ax.scatter(x, y, s=120, c='#FFD700', zorder=5, edgecolors='black', linewidths=0.8)
    ax.annotate(str(kpt_id), (x, y), textcoords='offset points',
                xytext=(4, 4), fontsize=7, color='white', fontweight='bold')

ax.set_title(f'Canonical Pitch Reference — {N_KEYPOINTS} Keypoints ({PITCH_WIDTH}×{PITCH_HEIGHT})',
             fontsize=13, pad=12)
ax.set_xlabel('x (canonical pixels)')
ax.set_ylabel('y (canonical pixels)')
ax.invert_yaxis()  # y=0 di atas seperti koordinat image
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/artifacts/results/figures/pitch_reference.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pitch visualization saved')

In [ ]:
# Cell 5: Analisis visibility distribution per keypoint
print('Menghitung visibility distribution... (mungkin beberapa menit)')

vis_counts = np.zeros((N_KEYPOINTS, 3), dtype=int)  # 3 = v0, v1, v2

for split in ['train', 'val', 'test']:
    lbl_dir = f'{YOLO_DIR}/{split}/labels'
    for fpath in glob.glob(f'{lbl_dir}/*.txt'):
        with open(fpath) as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                kpts_flat = parts[5:]
                for i in range(N_KEYPOINTS):
                    v = int(kpts_flat[i*3 + 2])
                    if 0 <= v <= 2:
                        vis_counts[i, v] += 1

# Plot stacked bar chart
fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(N_KEYPOINTS)
colors = ['#EF4444', '#F59E0B', '#22C55E']
labels = ['Invisible (v=0)', 'Occluded (v=1)', 'Visible (v=2)']
bottom = np.zeros(N_KEYPOINTS)
for v_idx in range(3):
    ax.bar(x, vis_counts[:, v_idx], bottom=bottom,
           color=colors[v_idx], label=labels[v_idx], alpha=0.85, edgecolor='white', linewidth=0.3)
    bottom += vis_counts[:, v_idx]
ax.set_xlabel('Keypoint ID')
ax.set_ylabel('Count')
ax.set_title('Visibility Distribution per Keypoint (all splits)')
ax.set_xticks(x)
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/artifacts/results/figures/visibility_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# Keypoint paling susah (sering invisible)
vis_pct = vis_counts[:, 2] / vis_counts.sum(axis=1).clip(1) * 100
print(f'\nTop 5 keypoint paling sering VISIBLE:')
for idx in np.argsort(vis_pct)[-5:][::-1]:
    print(f'  kpt_{idx:2d} ({kpt_names.get(idx, "?"):30s}): {vis_pct[idx]:.1f}% visible')
print(f'\nTop 5 keypoint paling sering INVISIBLE:')
for idx in np.argsort(vis_pct)[:5]:
    print(f'  kpt_{idx:2d} ({kpt_names.get(idx, "?"):30s}): {vis_pct[idx]:.1f}% visible')

In [ ]:
# Cell 6: Per-keypoint frequency (kemunculan di setiap gambar)
kpt_freq = np.zeros(N_KEYPOINTS, dtype=int)
total_images_counted = 0

for split in ['train', 'val', 'test']:
    lbl_dir = f'{YOLO_DIR}/{split}/labels'
    for fpath in glob.glob(f'{lbl_dir}/*.txt'):
        total_images_counted += 1
        with open(fpath) as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                kpts_flat = parts[5:]
                for i in range(N_KEYPOINTS):
                    v = int(kpts_flat[i*3 + 2])
                    if v > 0:  # hitung jika visible atau occluded
                        kpt_freq[i] += 1

freq_pct = kpt_freq / total_images_counted * 100

fig, ax = plt.subplots(figsize=(16, 4))
bars = ax.bar(np.arange(N_KEYPOINTS), freq_pct,
              color=['#3B82F6' if p > 50 else '#F87171' for p in freq_pct],
              edgecolor='white', linewidth=0.3)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.6, label='50% threshold')
ax.set_xlabel('Keypoint ID')
ax.set_ylabel('Frequency (% of images)')
ax.set_title('Keypoint Appearance Frequency — % of Images Containing Each Keypoint')
ax.set_xticks(np.arange(N_KEYPOINTS))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/artifacts/results/figures/keypoint_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total images scanned: {total_images_counted}')
print(f'Keypoints dengan frequency > 50%: {(freq_pct > 50).sum()}/{N_KEYPOINTS}')
print(f'Keypoints dengan frequency < 20%: {(freq_pct < 20).sum()}/{N_KEYPOINTS}  ← potensial bottleneck homography')

In [ ]:
# Cell 7: Visualisasi sample frame dengan keypoint overlay
import random
random.seed(42)

train_imgs = glob.glob(f'{YOLO_DIR}/train/images/*.jpg')
sample_imgs = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, sample_imgs):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt').replace('.png', '.txt')

    ax.imshow(img_rgb)
    ax.set_title(os.path.basename(img_path)[:30], fontsize=8)
    ax.axis('off')

    if not os.path.exists(label_path):
        continue

    with open(label_path) as f:
        line = f.readline().strip()
    if not line:
        continue

    parts = list(map(float, line.split()))
    kpts_flat = parts[5: 5 + N_KEYPOINTS*3]

    for i in range(N_KEYPOINTS):
        x_n = kpts_flat[i*3]
        y_n = kpts_flat[i*3 + 1]
        v   = int(kpts_flat[i*3 + 2])
        if v == 0: continue
        x_px = x_n * w
        y_px = y_n * h
        color = '#22C55E' if v == 2 else '#F59E0B'
        ax.scatter(x_px, y_px, s=25, c=color, zorder=5, linewidths=0)
        ax.annotate(str(i), (x_px, y_px), fontsize=5, color='white',
                    xytext=(2, 2), textcoords='offset points')

plt.suptitle('Sample Training Frames with Keypoint Annotations\n(green=visible, orange=occluded)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/artifacts/results/figures/sample_annotations.png',
            dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 8: Summary EDA untuk paper
print('='*55)
print('  EDA SUMMARY — paste ke Methods/Dataset section')
print('='*55)
for split, stats in total_stats.items():
    print(f'  {split.capitalize():5s}: {stats["n_images"]} images '
          f'(broadcast={stats["broadcast"]}, scouting={stats["scouting"]}, unknown={stats["unknown"]})')
print(f'  Total images   : {total_imgs}')
print(f'  Keypoints/img  : {N_KEYPOINTS}')
print(f'  Pitch canvas   : {PITCH_WIDTH} x {PITCH_HEIGHT} px')
print(f'  Avg vis rate   : {vis_pct.mean():.1f}% per keypoint')
print(f'  Min vis rate   : kpt_{vis_pct.argmin()} = {vis_pct.min():.1f}%')
print(f'  Max vis rate   : kpt_{vis_pct.argmax()} = {vis_pct.max():.1f}%')
print('='*55)
print('\nFigures saved ke artifacts/results/figures/')
print('  - pitch_reference.png')
print('  - visibility_distribution.png')
print('  - keypoint_frequency.png')
print('  - sample_annotations.png')